In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/yearly")

years = [2020, 2021, 2022, 2023, 2024, 2025]

yearly_files = {
    year: DATA_DIR / f"citysense_{year}.parquet"
    for year in years
}

for year, path in yearly_files.items():
    print(year, path)

2020 data\yearly\citysense_2020.parquet
2021 data\yearly\citysense_2021.parquet
2022 data\yearly\citysense_2022.parquet
2023 data\yearly\citysense_2023.parquet
2024 data\yearly\citysense_2024.parquet
2025 data\yearly\citysense_2025.parquet


In [2]:
train_parts = []

for year, path in yearly_files.items():
    df = pd.read_parquet(path)
    train_parts.append(df)
    print(year, df.shape)

final_data = pd.concat(train_parts, ignore_index=True)

print("Final dataset:", final_data.shape)

2020 (8309664, 16)
2021 (8286960, 16)
2022 (8286960, 16)
2023 (8286960, 16)
2024 (8309664, 16)
2025 (8286960, 16)
Final dataset: (49767168, 16)


In [3]:
final_data["crime_occurred"] = (
    final_data["crime_count"] >= 1
).astype("int8")

print(final_data["crime_occurred"].value_counts())
print("Crime rate:", final_data["crime_occurred"].mean())

crime_occurred
0    47103383
1     2663785
Name: count, dtype: int64
Crime rate: 0.05352494640643406


In [4]:
feature_columns = [
    "grid_id",
    "hour",
    "historical_grid_crime_count",
    "historical_grid_hour_crime_count",
    "day_of_week",
    "time_period",
    "historical_grid_day_crime_count",
    "historical_grid_time_period_crime_count",
    "is_weekend",
    "historical_grid_weekend_crime_count",
    "year",
    "month",
    "lat_grid",
    "lon_grid"
]

X_final = final_data[feature_columns]
y_final = final_data["crime_occurred"]

print("X:", X_final.shape)
print("y:", y_final.shape)

X: (49767168, 14)
y: (49767168,)


In [9]:
import xgboost as xgb

print(xgb.__version__)

3.4.1


In [11]:
import psutil

ram = psutil.virtual_memory()

print(f"RAM used: {ram.percent}%")
print(f"RAM available: {ram.available / 1024**3:.2f} GB")

RAM used: 66.0%
RAM available: 5.32 GB


In [12]:
import pandas as pd
import xgboost as xgb
from pathlib import Path

DATA_DIR = Path("data/yearly")

years = [2020, 2021, 2022, 2023, 2024, 2025]

yearly_files = {
    year: DATA_DIR / f"citysense_{year}.parquet"
    for year in years
}

feature_columns = [
    "grid_id",
    "hour",
    "historical_grid_crime_count",
    "historical_grid_hour_crime_count",
    "day_of_week",
    "time_period",
    "historical_grid_day_crime_count",
    "historical_grid_time_period_crime_count",
    "is_weekend",
    "historical_grid_weekend_crime_count",
    "year",
    "month",
    "lat_grid",
    "lon_grid"
]

for year, path in yearly_files.items():
    print(year, path.exists(), path)

2020 True data\yearly\citysense_2020.parquet
2021 True data\yearly\citysense_2021.parquet
2022 True data\yearly\citysense_2022.parquet
2023 True data\yearly\citysense_2023.parquet
2024 True data\yearly\citysense_2024.parquet
2025 True data\yearly\citysense_2025.parquet


In [13]:
class CrimeDataIter(xgb.DataIter):
    def __init__(self, files, feature_columns, batch_size=250_000):
        self.files = files
        self.feature_columns = feature_columns
        self.batch_size = batch_size
        self.file_index = 0
        self.reader = None
        super().__init__()

    def reset(self):
        self.file_index = 0
        self.reader = None

    def next(self, input_data):
        while True:
            if self.reader is None:
                if self.file_index >= len(self.files):
                    return False

                self.reader = pd.read_parquet(
                    self.files[self.file_index]
                )
                self.file_index += 1
                self.row_index = 0

            end = min(
                self.row_index + self.batch_size,
                len(self.reader)
            )

            batch = self.reader.iloc[self.row_index:end]
            self.row_index = end

            if len(batch) == 0:
                del self.reader
                self.reader = None
                continue

            X_batch = batch[self.feature_columns]
            y_batch = (batch["crime_count"] >= 1).astype("int8")

            input_data(
                data=X_batch,
                label=y_batch
            )

            return True

In [14]:
train_iter = CrimeDataIter(
    files=list(yearly_files.values()),
    feature_columns=feature_columns,
    batch_size=250_000
)

print("Iterator ready")

Iterator ready


In [15]:
train_iter.reset()

batch_data = None

def check_batch(data, label):
    global batch_data
    batch_data = (data, label)

train_iter.next(check_batch)

X_batch, y_batch = batch_data

print("X batch:", X_batch.shape)
print("y batch:", y_batch.shape)
print("Positive rate:", y_batch.mean())
print(X_batch.dtypes)

X batch: (250000, 14)
y batch: (250000,)
Positive rate: 0.003896
grid_id                                    category
hour                                           int8
historical_grid_crime_count                   int32
historical_grid_hour_crime_count              int32
day_of_week                                    int8
time_period                                category
historical_grid_day_crime_count               int32
historical_grid_time_period_crime_count       int32
is_weekend                                     int8
historical_grid_weekend_crime_count           int32
year                                          int64
month                                          int8
lat_grid                                    float32
lon_grid                                    float32
dtype: object


In [16]:
import pyarrow.parquet as pq

class CrimeDataIter(xgb.DataIter):
    def __init__(self, files, feature_columns, batch_size=250_000):
        self.files = files
        self.feature_columns = feature_columns
        self.batch_size = batch_size
        self.file_index = 0
        self.row_group_index = 0
        self.batch_data = None
        super().__init__()

    def reset(self):
        self.file_index = 0
        self.row_group_index = 0
        self.batch_data = None

    def next(self, input_data):
        while self.file_index < len(self.files):

            if self.batch_data is None:
                parquet_file = pq.ParquetFile(self.files[self.file_index])

                if self.row_group_index >= parquet_file.num_row_groups:
                    self.file_index += 1
                    self.row_group_index = 0
                    continue

                table = parquet_file.read_row_group(
                    self.row_group_index,
                    columns=self.feature_columns + ["crime_count"]
                )

                self.row_group_index += 1
                self.batch_data = table.to_pandas()

            batch = self.batch_data
            self.batch_data = None

            X_batch = batch[self.feature_columns]
            y_batch = (batch["crime_count"] >= 1).astype("int8")

            input_data(
                data=X_batch,
                label=y_batch
            )

            return True

        return False

In [17]:
train_iter = CrimeDataIter(
    files=list(yearly_files.values()),
    feature_columns=feature_columns,
    batch_size=250_000
)

print("Streaming iterator ready")

Streaming iterator ready


In [18]:
import pyarrow.parquet as pq

pf = pq.ParquetFile(yearly_files[2020])

print("Row groups:", pf.num_row_groups)
print("Rows in first row group:", pf.metadata.row_group(0).num_rows)

Row groups: 10
Rows in first row group: 878400


In [19]:
class CrimeDataIter(xgb.DataIter):
    def __init__(self, files, feature_columns, batch_size=250_000):
        self.files = files
        self.feature_columns = feature_columns
        self.batch_size = batch_size
        self.file_index = 0
        self.row_group_index = 0
        self.batch_data = None
        self.batch_start = 0
        self.parquet_file = None
        super().__init__()

    def reset(self):
        self.file_index = 0
        self.row_group_index = 0
        self.batch_data = None
        self.batch_start = 0
        self.parquet_file = None

    def next(self, input_data):
        while True:

            if self.batch_data is None:
                if self.parquet_file is None:
                    if self.file_index >= len(self.files):
                        return False

                    self.parquet_file = pq.ParquetFile(
                        self.files[self.file_index]
                    )

                if self.row_group_index >= self.parquet_file.num_row_groups:
                    self.file_index += 1
                    self.row_group_index = 0
                    self.parquet_file = None
                    continue

                table = self.parquet_file.read_row_group(
                    self.row_group_index,
                    columns=self.feature_columns + ["crime_count"]
                )

                self.batch_data = table.to_pandas()
                self.batch_start = 0
                self.row_group_index += 1

            end = min(
                self.batch_start + self.batch_size,
                len(self.batch_data)
            )

            batch = self.batch_data.iloc[self.batch_start:end]
            self.batch_start = end

            if self.batch_start >= len(self.batch_data):
                self.batch_data = None

            X_batch = batch[self.feature_columns]
            y_batch = (batch["crime_count"] >= 1).astype("int8")

            input_data(
                data=X_batch,
                label=y_batch
            )

            return True

In [20]:
train_iter = CrimeDataIter(
    files=list(yearly_files.values()),
    feature_columns=feature_columns,
    batch_size=250_000
)

print("Iterator updated")

Iterator updated


In [21]:
train_iter.reset()

batch_data = None

def check_batch(data, label):
    global batch_data
    batch_data = (data, label)

train_iter.next(check_batch)

X_batch, y_batch = batch_data

print("X batch:", X_batch.shape)
print("y batch:", y_batch.shape)
print("Positive rate:", y_batch.mean())

X batch: (250000, 14)
y batch: (250000,)
Positive rate: 0.003896


In [22]:
import gc
import psutil

del X_batch
del y_batch
del batch_data

gc.collect()

ram = psutil.virtual_memory()

print(f"RAM used: {ram.percent}%")
print(f"RAM available: {ram.available / 1024**3:.2f} GB")

RAM used: 75.6%
RAM available: 3.81 GB


In [23]:
final_model = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=600,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=20,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

print("Final model ready")

Final model ready


In [25]:
dtrain = xgb.QuantileDMatrix(
    train_iter,
    enable_categorical=True
)

print("DMatrix created")

XGBoostError: [18:29:08] C:\actions-runner\_work\xgboost\xgboost\src\data\quantile_dmatrix.cc:135: Check failed: cats.n_total_cats == p_info->cats->NumCatsTotal() (50 vs. 104) : Inconsistent number of categories between batches.

In [26]:
GRID_CATEGORIES = pd.read_parquet(
    yearly_files[2020],
    columns=["grid_id"]
)["grid_id"].cat.categories

TIME_CATEGORIES = pd.read_parquet(
    yearly_files[2020],
    columns=["time_period"]
)["time_period"].cat.categories

In [27]:
class CrimeDataIter(xgb.DataIter):
    def __init__(self, files, feature_columns, batch_size=250_000):
        self.files = files
        self.feature_columns = feature_columns
        self.batch_size = batch_size
        self.file_index = 0
        self.row_group_index = 0
        self.batch_data = None
        self.batch_start = 0
        self.parquet_file = None
        super().__init__()

    def reset(self):
        self.file_index = 0
        self.row_group_index = 0
        self.batch_data = None
        self.batch_start = 0
        self.parquet_file = None

    def next(self, input_data):
        while True:
            if self.batch_data is None:
                if self.parquet_file is None:
                    if self.file_index >= len(self.files):
                        return False

                    self.parquet_file = pq.ParquetFile(
                        self.files[self.file_index]
                    )

                if self.row_group_index >= self.parquet_file.num_row_groups:
                    self.file_index += 1
                    self.row_group_index = 0
                    self.parquet_file = None
                    continue

                table = self.parquet_file.read_row_group(
                    self.row_group_index,
                    columns=self.feature_columns + ["crime_count"]
                )

                self.batch_data = table.to_pandas()
                self.batch_start = 0
                self.row_group_index += 1

            end = min(
                self.batch_start + self.batch_size,
                len(self.batch_data)
            )

            batch = self.batch_data.iloc[self.batch_start:end]
            self.batch_start = end

            if self.batch_start >= len(self.batch_data):
                self.batch_data = None

            batch["grid_id"] = pd.Categorical(
                batch["grid_id"],
                categories=GRID_CATEGORIES
            )

            batch["time_period"] = pd.Categorical(
                batch["time_period"],
                categories=TIME_CATEGORIES
            )

            X_batch = batch[self.feature_columns]
            y_batch = (batch["crime_count"] >= 1).astype("int8")

            input_data(
                data=X_batch,
                label=y_batch
            )

            return True

In [28]:
train_iter = CrimeDataIter(
    files=list(yearly_files.values()),
    feature_columns=feature_columns,
    batch_size=250_000
)

print("Iterator fixed")

Iterator fixed


In [29]:
train_iter.reset()

dtrain = xgb.QuantileDMatrix(
    train_iter,
    enable_categorical=True
)

print("DMatrix created successfully")

DMatrix created successfully


In [30]:
final_model = xgb.train(
    {
        "objective": "binary:logistic",
        "learning_rate": 0.05,
        "max_depth": 8,
        "min_child_weight": 20,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "tree_method": "hist",
        "enable_categorical": True,
        "max_bin": 256,
        "n_jobs": -1,
        "seed": 42
    },
    dtrain,
    num_boost_round=600
)

C:\Users\sri16\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:34:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "enable_categorical" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [31]:
print("Training complete")
print("Number of trees:", final_model.num_boosted_rounds())

Training complete
Number of trees: 600


In [33]:
sample_df = pd.read_parquet(
    yearly_files[2025],
    columns=feature_columns
).head(10)

predictions = final_model.predict(
    xgb.DMatrix(sample_df, enable_categorical=True)
)

print(predictions)

[0.00061638 0.00033548 0.00032332 0.00034417 0.00026895 0.00023321
 0.00030887 0.00037215 0.0004213  0.00040561]


In [34]:
final_model.save_model("citysense_final_model.json")

print("Model saved successfully")

Model saved successfully


In [35]:
from pathlib import Path

model_path = Path("citysense_final_model.json")

print("Exists:", model_path.exists())
print("Size:", model_path.stat().st_size / 1024**2, "MB")

Exists: True
Size: 51.55933856964111 MB
